<a href="https://colab.research.google.com/github/Shineii86/MoeStickerBot/blob/main/notebooks/MoeStickerBotV3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <pre style="background:#0C0C0C; color:#00FF00; padding:10px; border-radius:8px; font-family:'Courier New',monospace;">
╔══════════════════════════════════════════════════════════╗
║  ███╗   ███╗ ██████╗ ███████╗    ███████╗████████╗██╗ ██████╗██╗  ██╗███████╗██████╗   ║
║  ████╗ ████║██╔═══██╗██╔════╝    ██╔════╝╚══██╔══╝██║██╔════╝██║ ██╔╝██╔════╝██╔══██╗  ║
║  ██╔████╔██║██║   ██║█████╗      ███████╗   ██║   ██║██║     █████╔╝ █████╗  ██████╔╝  ║
║  ██║╚██╔╝██║██║   ██║██╔══╝      ╚════██║   ██║   ██║██║     ██╔═██╗ ██╔══╝  ██╔══██╗  ║
║  ██║ ╚═╝ ██║╚██████╔╝███████╗    ███████║   ██║   ██║╚██████╗██║  ██╗███████╗██║  ██║  ║
║  ╚═╝     ╚═╝ ╚═════╝ ╚══════╝    ╚══════╝   ╚═╝   ╚═╝ ╚═════╝╚═╝  ╚═╝╚══════╝╚═╝  ╚═╝  ║
╠══════════════════════════════════════════╣
║               TERMINAL EDITION — V3                       ║
╚══════════════════════════════════════════╝
  </pre>
</div>

```bash
➜ root@shinei ~ # cat README.md
```

> **Telegram Sticker Bot self-hosted in Google Colab**  
> Import LINE/Kakao · Create · Manage · Export — all from your terminal.

---

In [ ]:
#@title 🖥️ 1. Setup Environment & Build Bot (Run Once)

import sys, time, subprocess, os, urllib.request, json, requests
from itertools import cycle
from tqdm.notebook import tqdm

# ========== TERMINAL UI ENGINE ==========
class T:
    R = '\033[0m'; BOLD = '\033[1m'; DIM = '\033[2m'
    BLK = '\033[30m'; RED = '\033[31m'; GRN = '\033[32m'; YLW = '\033[33m'
    BLU = '\033[34m'; MAG = '\033[35m'; CYN = '\033[36m'; WHT = '\033[37m'
    BBLK = '\033[90m'; BRED = '\033[91m'; BGRN = '\033[92m'; BYLW = '\033[93m'
    BBLU = '\033[94m'; BMAG = '\033[95m'; BCYN = '\033[96m'; BWHT = '\033[97m'
    BGBLK = '\033[40m'; BGRED = '\033[41m'; BGGRN = '\033[42m'; BGYLW = '\033[43m'
    BGBLU = '\033[44m'; BGMAG = '\033[45m'; BGCYN = '\033[46m'; BGWHT = '\033[47m'
    BGBBLK = '\033[100m'; BGBRED = '\033[101m'; BGBGRN = '\033[102m'; BGBYLW = '\033[103m'

PROMPT = f"{T.BOLD}{T.BGRN}➜{T.R} {T.BCYN}root@shinei{T.R} {T.BOLD}{T.BLU}~{T.R}{T.BOLD}#{T.R} "
def cmd(t): term_print(f"{PROMPT}{T.BWHT}{t}{T.R}")
def success(m): term_print(f"{T.BOLD}{T.BGGRN}{T.BLK} ✓ {m} {T.R}")
def error(m): term_print(f"{T.BOLD}{T.BGRED}{T.WHT} ✗ {m} {T.R}")
def info(m): term_print(f"{T.BOLD}{T.BGBLU}{T.WHT} ℹ {m} {T.R}")
def warn(m): term_print(f"{T.BOLD}{T.BGYLW}{T.BLK} ⚠ {m} {T.R}")
def header(t): term_print(f"\n{T.BOLD}{T.BGBLU}{T.WHT}═══ {t} {T.R}{'═'*(50-len(t))}")
def term_print(t, end='\n'): sys.stdout.write(t+end); sys.stdout.flush()
def spinner(msg, dur=2):
    frames = cycle(['⠋','⠙','⠹','⠸','⠼','⠴','⠦','⠧','⠇','⠏'])
    end=time.time()+dur
    while time.time()<end:
        sys.stdout.write(f'\r{T.BCYN}{next(frames)} {msg}{T.R}  '); sys.stdout.flush(); time.sleep(0.1)
    sys.stdout.write(f'\r{T.BGRN}✔{T.R} {msg}   \n')

term_print(f"{T.BOLD}{T.BGBLU}{T.WHT} TERMINAL UI INITIALIZED {T.R}")

# ========== INSTALL DEPENDENCIES ==========
header("SYSTEM DEPENDENCIES")
cmd("apt-get update -qq && apt-get install -y -qq imagemagick libarchive-tools ffmpeg curl gifsicle python3 exiv2")
!apt-get update -qq && apt-get install -y -qq imagemagick libarchive-tools ffmpeg curl gifsicle python3 exiv2
success("Core packages installed")

# Go
url = "https://go.dev/dl/go1.21.5.linux-amd64.tar.gz"
cmd(f"wget -q {url}")
with tqdm(unit='B', unit_scale=True, desc=f"{T.BCYN}Downloading Go{T.R}") as t:
    urllib.request.urlretrieve(url, "go.tar.gz", reporthook=lambda b,bs,total: t.update(b*bs-t.n))
cmd("tar -C /usr/local -xzf go.tar.gz")
!tar -C /usr/local -xzf go.tar.gz
os.environ['PATH'] += ":/usr/local/go/bin"
os.environ['GOPATH'] = "/root/go"
os.environ['GO111MODULE'] = "on"
!mkdir -p $GOPATH
success(f"Go {subprocess.getoutput('go version').split()[2]} installed")

# ========== PYTHON HELPERS ==========
header("PYTHON HELPERS")
for f,desc in [("msb_emoji.py","Emoji"), ("msb_kakao_decrypt.py","Kakao"), ("msb_rlottie.py","Lottie")]:
    cmd(f"wget -q -O /usr/local/bin/{f} https://raw.githubusercontent.com/star-39/moe-sticker-bot/master/tools/{f}")
    !wget -q https://raw.githubusercontent.com/star-39/moe-sticker-bot/master/tools/{f} -O /usr/local/bin/{f}
    !chmod +x /usr/local/bin/{f}
    term_print(f"  {T.BGRN}✓{T.R} {desc}")
success("Helpers installed")

# ========== BUILD BOT ==========
header("BUILDING MOE-STICKER-BOT")
!rm -rf moe-sticker-bot
cmd("git clone --depth 1 https://github.com/star-39/moe-sticker-bot.git")
!git clone --depth 1 https://github.com/star-39/moe-sticker-bot.git 2>&1 | grep -v "Cloning"
%cd moe-sticker-bot
spinner("Downloading Go modules", 2)
!go mod download
spinner("Compiling binary", 3)
!go build -o moe-sticker-bot cmd/moe-sticker-bot/main.go
if os.path.exists("moe-sticker-bot"):
    sz = os.path.getsize("moe-sticker-bot")/1024/1024
    success(f"Build complete — Binary: {sz:.1f} MB")
else:
    error("Build failed")


In [ ]:
#@title ⚙️ 2. Configure & Launch Bot

# ========== CONFIGURATION ==========
BOT_TOKEN = ""  #@param {type:"string"}
ENABLE_DB = False  #@param {type:"boolean"}
DB_ADDR = "localhost:3306"  #@param {type:"string"}
DB_USER = "moe_bot"  #@param {type:"string"}
DB_PASS = ""  #@param {type:"string"}
DB_NAME = "moe_sticker_bot"  #@param {type:"string"}
ENABLE_WEBAPP = False  #@param {type:"boolean"}
WEBAPP_PORT = 8080  #@param {type:"integer"}
NGROK_AUTHTOKEN = ""  #@param {type:"string"}
DATA_DIR = "moe_sticker_bot_data"  #@param {type:"string"}
LOG_LEVEL = "info"  #@param ["debug", "info", "warn", "error"]
HTTP_PROXY = ""  #@param {type:"string"}

header("CONFIGURATION")
if BOT_TOKEN:
    term_print(f"  {T.BGRN}BOT_TOKEN{T.R} = {BOT_TOKEN[:8]}...{BOT_TOKEN[-4:]}")
else:
    warn("BOT_TOKEN missing!")

if ENABLE_WEBAPP and not NGROK_AUTHTOKEN:
    warn("WebApp enabled but no ngrok token — disabled")
    ENABLE_WEBAPP = False

# ========== NGROK (if enabled) ==========
WEBAPP_URL = ""
if ENABLE_WEBAPP:
    header("NGROK TUNNEL")
    if not os.path.exists("./ngrok"):
        !wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz && tar -xzf ngrok*.tgz && chmod +x ngrok
    !./ngrok config add-authtoken {NGROK_AUTHTOKEN}
    !pkill -f ngrok || true
    ngrok_proc = subprocess.Popen(["./ngrok", "http", str(WEBAPP_PORT), "--log", "stdout"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    for _ in range(10):
        try:
            r = requests.get("http://127.0.0.1:4040/api/tunnels")
            if r.status_code==200:
                tuns = r.json()['tunnels']
                if tuns:
                    WEBAPP_URL = tuns[0]['public_url']
                    success(f"ngrok: {WEBAPP_URL}")
                    break
        except: pass
        time.sleep(1)
    else:
        error("ngrok URL failed")
        ENABLE_WEBAPP = False

# ========== LAUNCH BOT ==========
header("LAUNCHING BOT")
if not BOT_TOKEN:
    error("No BOT_TOKEN. Aborting.")
    sys.exit(1)

cmd_line = ["./moe-sticker-bot", f"--bot_token={BOT_TOKEN}", f"--log_level={LOG_LEVEL}", f"--data_dir={DATA_DIR}"]
if ENABLE_DB and DB_ADDR:
    cmd_line.extend([f"--db_addr={DB_ADDR}", f"--db_user={DB_USER}", f"--db_pass={DB_PASS}", f"--db_name={DB_NAME}"])
if ENABLE_WEBAPP and WEBAPP_URL:
    cmd_line.append(f"--webapp_url={WEBAPP_URL}")
    cmd_line.append(f"--webapp_listen_addr=0.0.0.0:{WEBAPP_PORT}")
if HTTP_PROXY:
    cmd_line.append(f"--http_proxy={HTTP_PROXY}")

cmd(" ".join(cmd_line).replace(BOT_TOKEN, "[REDACTED]"))
log_out = open("bot_stdout.log", "w")
log_err = open("bot_stderr.log", "w")
process = subprocess.Popen(cmd_line, stdout=log_out, stderr=log_err)
spinner("Starting bot", 3)
time.sleep(2)

if process.poll() is None:
    success(f"Bot RUNNING — PID {process.pid}")
    term_print(f"{T.BOLD}{T.BGGRN}{T.BLK} 📱 Send /start on Telegram! {T.R}")
    if WEBAPP_URL:
        term_print(f"{T.BOLD}{T.BGCYN}{T.BLK} 🌐 WebApp: {WEBAPP_URL} {T.R}")
else:
    error("Bot exited. Check logs:")
    !cat bot_stderr.log


In [ ]:
#@title 📜 3. Monitor & Control

ACTION = "View Logs"  #@param ["View Logs", "Stop Bot"]
LOG_TYPE = "stderr"  #@param ["stdout", "stderr"]
LINES = 30  #@param {type:"slider", min:10, max:100, step:10}

if ACTION == "View Logs":
    cmd(f"tail -n {LINES} bot_{LOG_TYPE}.log")
    !tail -n {LINES} bot_{LOG_TYPE}.log
else:
    header("SHUTDOWN")
    !pkill -f moe-sticker-bot && term_print(f"{T.BGRED}Bot terminated{T.R}") || term_print(f"{T.YLW}No bot running{T.R}")
    !pkill -f ngrok && term_print(f"{T.BGRED}ngrok terminated{T.R}") || term_print(f"{T.YLW}No ngrok running{T.R}")
    success("Cleanup complete")


---
<div align="center">
<pre style="background:#0C0C0C; color:#00FF00; padding:10px; border-radius:8px; font-family:'Courier New',monospace;">
╔══════════════════════════════════════════╗
║                     SESSION TERMINATED                    ║
╚══════════════════════════════════════════╝
</pre>
<p style="font-family:monospace;">
➜ root@shinei ~ # <span style="color:#00FF00">exit</span><br>
Connection to colab closed.
</p>
</div>